In [13]:
# Load the cleaned dataset safely
import pandas as pd
import numpy as np

df = pd.read_csv(
    "cleaned_arxiv_datasets.csv",
    engine="python",
    on_bad_lines="skip"
)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst Five Rows:")
display(df.head())

Dataset loaded successfully
Dataset shape: (287421, 10)

Column Names:
['id', 'title', 'category', 'category_code', 'published_date', 'updated_date', 'authors', 'first_author', 'summary', 'summary_word_count']

First Five Rows:


,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count
0,cs-9308101v1,Dynamic Backtracking,Artificial Intelligence,cs.AI,1993-08-01,1993-08-01,['M. L. Ginsberg'],'M. L. Ginsberg',Because of their occasional need to return to ...,79
1,cs-9308102v1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,cs.AI,1993-08-01,1993-08-01,['M. P. Wellman'],'M. P. Wellman',Market price systems constitute a well-underst...,119
2,cs-9309101v1,An Empirical Analysis of Search in GSAT,Artificial Intelligence,cs.AI,1993-09-01,1993-09-01,"['I. P. Gent', 'T. Walsh']",'I. P. Gent',We describe an extensive study of search in GS...,167
3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,cs.AI,1993-11-01,1993-11-01,"['F. Bergadano', 'D. Gunetti', 'U. Trinchero']",'F. Bergadano',As real logic programmers normally use cut (!)...,174
4,cs-9311102v1,Software Agents: Completing Patterns and Const...,Artificial Intelligence,cs.AI,1993-11-01,1993-11-01,"['J. C. Schlimmer', 'L. A. Hermens']",'J. C. Schlimmer',To support the goal of allowing users to recor...,187


In [14]:
# Check that the required text fields are complete
df = df.dropna(subset=["title", "summary"]).copy()

# Combine the title and summary
df["combined_text"] = (
    df["title"].str.strip()
    + " "
    + df["summary"].str.strip()
)

print("Combined text created successfully!\n")

display(
    df[["title", "summary", "combined_text"]].head()
)

Combined text created successfully!



,title,summary,combined_text
0,Dynamic Backtracking,Because of their occasional need to return to ...,Dynamic Backtracking Because of their occasion...
1,A Market-Oriented Programming Environment and ...,Market price systems constitute a well-underst...,A Market-Oriented Programming Environment and ...
2,An Empirical Analysis of Search in GSAT,We describe an extensive study of search in GS...,An Empirical Analysis of Search in GSAT We des...
3,The Difficulties of Learning Logic Programs wi...,As real logic programmers normally use cut (!)...,The Difficulties of Learning Logic Programs wi...
4,Software Agents: Completing Patterns and Const...,To support the goal of allowing users to recor...,Software Agents: Completing Patterns and Const...


In [15]:


# Convert the combined text to lowercase
df["combined_text"] = df["combined_text"].str.lower()

print("Text converted to lowercase successfully!\n")


display(df[["combined_text"]].head())

Text converted to lowercase successfully!



,combined_text
0,dynamic backtracking because of their occasion...
1,a market-oriented programming environment and ...
2,an empirical analysis of search in gsat we des...
3,the difficulties of learning logic programs wi...
4,software agents: completing patterns and const...


In [16]:
# Remove punctuation

import re

df["combined_text"] = df["combined_text"].apply(
    lambda text: re.sub(r"[^\w\s]", " ", str(text))
)

# Remove extra spaces created after punctuation removal
df["combined_text"] = (
    df["combined_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Punctuation removed successfully!\n")

display(df[["combined_text"]].head())

Punctuation removed successfully!



,combined_text
0,dynamic backtracking because of their occasion...
1,a market oriented programming environment and ...
2,an empirical analysis of search in gsat we des...
3,the difficulties of learning logic programs wi...
4,software agents completing patterns and constr...


In [17]:

# Remove Stopwords


import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')

# Load English stopwords
stop_words = set(stopwords.words("english"))

# Remove stopwords from the combined text
df["combined_text"] = df["combined_text"].apply(
    lambda text: " ".join(
        word for word in str(text).split()
        if word.lower() not in stop_words
    )
)

print("stopwords removed successfully!\n")

# Display the first 5 rows
display(df[["combined_text"]].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


stopwords removed successfully!



,combined_text
0,dynamic backtracking occasional need return sh...
1,market oriented programming environment applic...
2,empirical analysis search gsat describe extens...
3,difficulties learning logic programs cut real ...
4,software agents completing patterns constructi...


In [18]:


import nltk
from nltk.stem import WordNetLemmatizer
from tqdm.auto import tqdm

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def fast_lemmatize(text):
    words = str(text).split()

    cleaned_words = [
        lemmatizer.lemmatize(
            lemmatizer.lemmatize(word, pos="v"),
            pos="n"
        )
        for word in words
    ]

    return " ".join(cleaned_words)

# Show progress while processing
tqdm.pandas(desc="Lemmatizing text")

df["cleaned_text"] = (
    df["combined_text"]
    .astype(str)
    .progress_apply(fast_lemmatize)
)

print("\nLemmatization completed successfully!")

display(
    df[["combined_text", "cleaned_text"]].head()
)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Lemmatizing text:   0%|          | 0/287421 [00:00<?, ?it/s]


Lemmatization completed successfully!


,combined_text,cleaned_text
0,dynamic backtracking occasional need return sh...,dynamic backtrack occasional need return shall...
1,market oriented programming environment applic...,market orient program environment application ...
2,empirical analysis search gsat describe extens...,empirical analysis search gsat describe extens...
3,difficulties learning logic programs cut real ...,difficulty learn logic program cut real logic ...
4,software agents completing patterns constructi...,software agent complete pattern construct user...


In [19]:
# Validate the final cleaned text

missing_cleaned_text = df["cleaned_text"].isna().sum()

empty_cleaned_text = (
    df["cleaned_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Number of missing cleaned_text rows:",
    missing_cleaned_text
)

print(
    "Number of empty cleaned_text rows:",
    empty_cleaned_text
)

print("Final preprocessed dataset shape:", df.shape)

Number of missing cleaned_text rows: 0
Number of empty cleaned_text rows: 0
Final preprocessed dataset shape: (287421, 12)


In [20]:
df.to_csv(
    "preprocessed_dataset.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")
print("Saved dataset shape:", df.shape)

Preprocessed dataset saved successfully!
Saved dataset shape: (287421, 12)
